In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/celoy/SoftMobility.git
except ImportError:
    pass

# Mobility Matrix of a Rigid Assembly

Testing the `SoftBody` class with no degrees-of-freedom.

The **mobility matrix** $\mathbf{M}$ of an assembly relates applied forces and torques $(\mathbf{F}, \mathbf{T})$ to the resulting translational and rotational velocities $(\mathbf{U}, \boldsymbol{\Omega})$ in Stokes flow:

$$\begin{pmatrix}\mathbf{U}\\ \boldsymbol{\Omega}\end{pmatrix} = \mathbf{M} \begin{pmatrix}\mathbf{F}\\ \mathbf{T}\end{pmatrix}$$

For an assembly of $N$ beads, the library builds a $6N \times 6N$ **grand mobility matrix** using the Rotne–Prager–Yamakawa (RPY) approximation, which captures pairwise hydrodynamic interactions between beads. This tutorial shows how to reduce it to a single $6 \times 6$ effective mobility for a **rigid** assembly (no internal degrees of freedom), and how to locate the special reference points that simplify its structure.

In [1]:
import numpy as np

import softmobility as sm

## Input with YAML format

The assembly consists of three spheres of different radii arranged asymmetrically. Because no `dof_names` are declared, the body is **rigid**: it has no internal degrees of freedom.

The `xref` design parameters define the **reference point** of the assembly. Since all sphere positions are expressed in terms of `xref`, varying these parameters rigidly translates the entire assembly — effectively shifting which point in space serves as the origin for forces and torques.

In [2]:
yaml_data = """
design_names:
    - radius
    - xref

defaults:
    radius0: 1.
    radius1: 0.75
    radius2: 0.5
    xref0: 0.0
    xref1: 0.0
    xref2: 0.0

spheres:
  - # sm.Sphere 0
    radius: radius0
    position: [xref0, xref1, xref2]

  - # sm.Sphere 1
    radius: radius1
    position: [xref0 - 0.33 + radius0 + radius1, xref1, xref2]

  - # sm.Sphere 2
    radius: radius2
    position: [xref0, xref1 + radius0 + radius2, xref2]
"""

In [3]:
# Creating sm.SoftBody object
mybody = sm.SoftBody(yaml_data)

  Found variables: radius0, radius1, radius2, xref0, xref1, xref2, 
    Sphere 0
      Radius: radius0
      Position: ['xref0', 'xref1', 'xref2']
      Orientation: ['0', '0', '0']
      Force: ['0', '0', '0']
      Torque: ['0', '0', '0']
    Sphere 1
      Radius: radius1
      Position: ['radius0 + radius1 + xref0 - 0.33', 'xref1', 'xref2']
      Orientation: ['0', '0', '0']
      Force: ['0', '0', '0']
      Torque: ['0', '0', '0']
    Sphere 2
      Radius: radius2
      Position: ['xref0', 'radius0 + radius2 + xref1', 'xref2']
      Orientation: ['0', '0', '0']
      Force: ['0', '0', '0']
      Torque: ['0', '0', '0']


## Calculating the rigid mobility matrix

The method `compute_grand_mobility()` evaluates the $6N \times 6N$ grand mobility tensor using the RPY approximation. For $N = 3$ spheres this yields an $18 \times 18$ matrix coupling all translational and rotational degrees of freedom of every bead.

In [4]:
M_grand = mybody.compute_grand_mobility()
print('Grand mobility tensor M shape:', M_grand.shape)
print('Grand mobility tensor:', M_grand)

Grand mobility tensor M shape: (18, 18)
Grand mobility tensor: [[ 0.05305165  0.          0.          0.          0.          0.
   0.0415653   0.          0.          0.          0.          0.
   0.03143802  0.          0.          0.          0.          0.01768388]
 [ 0.          0.05305165  0.          0.          0.          0.
   0.          0.03525783  0.         -0.         -0.         -0.01973256
   0.          0.04322727  0.          0.          0.          0.        ]
 [ 0.          0.          0.05305165  0.          0.          0.
   0.          0.          0.03525783  0.          0.01973256  0.
   0.          0.          0.03143802 -0.01768388 -0.         -0.        ]
 [ 0.          0.          0.          0.03978873  0.          0.
   0.          0.         -0.          0.01389617 -0.         -0.
  -0.          0.          0.01768388 -0.00589463 -0.          0.        ]
 [ 0.          0.          0.          0.          0.03978873  0.
   0.          0.         -0.019732

The method `compute_rigid_mobility()` returns directly the $6 \times 6$ effective mobility of the assembly, treating it as perfectly rigid: it maps a force and torque applied at the assembly reference point $O$ to the resulting translational and rotational velocities.

In [5]:
M_rigid = np.array(mybody.compute_rigid_mobility())

print('Rigid-body mobility matrix (forces/torques at ref point O):')
print(M_rigid)

Rigid-body mobility matrix (forces/torques at ref point O):
[[ 0.04604208 -0.00215553  0.          0.          0.          0.00394097]
 [-0.00215553  0.04667006  0.          0.          0.         -0.0067573 ]
 [ 0.          0.          0.04540306 -0.00594933  0.00820442  0.        ]
 [ 0.          0.         -0.00594933  0.02218566 -0.00138137  0.        ]
 [ 0.          0.          0.00820442 -0.00138137  0.01909124  0.        ]
 [ 0.00394098 -0.0067573   0.          0.          0.          0.01556045]]


## Assert the properties of the mobility matrix

The mobility matrix has two properties that we can check:

- **Symmetry**: a consequence of the Lorentz reciprocal theorem in Stokes flow: swapping the roles of force and velocity leaves the power dissipation invariant.
- **Positive definiteness**: ensures that viscous drag always dissipates energy: $\mathbf{F}^\top \mathbf{M}^{-1} \mathbf{F} \geq 0$ for all $\mathbf{F} \neq \mathbf{0}$.

In [6]:
# Check if the matrix is symmetric
assert np.allclose(M_rigid, M_rigid.T), "M_mean is not symmetric"

# Check if the matrix is positive semi-definite (all eigenvalues >= 0)
eigvals = np.linalg.eigvalsh(M_rigid)
print("Eigenvalues of M:", eigvals)
assert np.all(eigvals >= 0), "M_mean is not positive definite"

print("The rigid mobility matrix is symmetric definite positive (as expected)")

Eigenvalues of M: [0.01379064 0.01672019 0.02082717 0.04422669 0.04913259 0.05025527]
The rigid mobility matrix is symmetric definite positive (as expected)


## Hydrodynamic centers

The $6 \times 6$ mobility and resistance matrices have the block structure

$$\mathbf{M} = \begin{pmatrix} \mathbf{a} & \mathbf{b}^\top \\ \mathbf{b} & \mathbf{c} \end{pmatrix}, \qquad \mathbf{R} \equiv \mathbf{M}^{-1} = \begin{pmatrix} \mathbf{A} & \mathbf{B}^\top \\ \mathbf{B} & \mathbf{C} \end{pmatrix},$$

where the $3\times 3$ off-diagonal blocks $\mathbf{b}$ and $\mathbf{B}$ couple translation to rotation. Their asymmetry with respect to an arbitrary reference point reflects the choice of origin.

Choosing a special reference point for forces and torques makes the coupling block symmetric. This special point is called the **hydrodynamic center**. Two such centers exist (Kim & Karrila, Secs. 5.2.2 & 5.3.2):

- **Center of resistance** $\mathbf{x}_{CR}$: symmetrizes $\mathbf{B}$ in the resistance tensor.
$$\mathbf{x}_{CR} = -\tfrac{1}{2}\bigl(\mathrm{tr}(\mathbf{A})\,\mathbf{I} - \mathbf{A}\bigr)^{-1} \cdot \boldsymbol{\epsilon} : (\mathbf{B} - \mathbf{B}^\top)$$

- **Center of mobility** $\mathbf{x}_{CM}$: symmetrizes $\mathbf{b}$ in the mobility tensor.
$$\mathbf{x}_{CM} = \tfrac{1}{2}\bigl(\mathrm{tr}(\mathbf{c})\,\mathbf{I} - \mathbf{c}\bigr)^{-1} \cdot \boldsymbol{\epsilon} : (\mathbf{b} - \mathbf{b}^\top)$$

where $\boldsymbol{\epsilon}$ is the Levi-Civita tensor.

Both formulas give the position of the centre **relative to the current reference point**. To make that centre the new reference, the body must therefore be translated by $-\mathbf{x}_{CR}$ (resp. $-\mathbf{x}_{CM}$) — equivalently, `xref` is set to the opposite of the computed centre.

In [7]:
# Tensors to compute the mobility center
b = M_rigid[3:, :3]
c = M_rigid[3:, 3:]

# Tensors to compute the resistance center
resistance_matrix = np.linalg.inv(M_rigid)
A = resistance_matrix[:3, :3]
B = resistance_matrix[3:, :3]

levicivita = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]],
        [[0.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
        [[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# From Kim & Karrila 5.2.2 and 5.3.2
trAItmA_inv = np.linalg.inv(np.trace(A) * np.eye(3) - A)
x_cr = -0.5 * np.einsum("ij,jkl,kl->i", trAItmA_inv, levicivita, B - B.transpose())
trcItmc_inv = np.linalg.inv(np.trace(c) * np.eye(3) - c)
x_cm = 0.5 * np.einsum("ij,jkl,kl->i", trcItmc_inv, levicivita, b - b.transpose())

print("Hydrodynamic center of resistance:", x_cr)
print("Hydrodynamic center of mobility:", x_cm)

Hydrodynamic center of resistance: [ 0.42279189  0.24754643 -0.        ]
Hydrodynamic center of mobility: [0.42194482 0.24658029 0.        ]


We shift the assembly so that its centre of resistance coincides with the reference point, by setting `xref` to $-\mathbf{x}_{CR}$. Since all sphere positions are defined relative to `xref`, this rigidly translates the whole assembly. The rotation–translation block of the **resistance** tensor should now be symmetric.

In [8]:
# Assert the resistance tensor with x_cr the reference point
print("BEFORE change of coordinates:")
R_mean = np.linalg.inv(M_rigid)
R_translation_rotation = R_mean[3:,:3]
print("Rotation-translation component of the resistance tensor:")
print(R_translation_rotation, "\n")


# Shift the assembly so the centre of resistance coincides with the reference point
mybody.set_design_defaults(new_dict={"xref0": -x_cr[0], "xref1": -x_cr[1], "xref2": -x_cr[2]})

M_rigid = np.array(mybody.compute_rigid_mobility())
R_mean = np.linalg.inv(M_rigid)

print("\nAFTER change of coordinates:")
R_translation_rotation = R_mean[3:,:3]
print("Rotation-translation component of the resistance tensor:")
print(R_translation_rotation)

# Check if the matrix is symmetric
if np.allclose(R_translation_rotation, R_translation_rotation.T, atol=1E-5):
    print("R_mean is now symmetric")
else:
    raise ValueError("R_mean is not symmetric after change of coordinates")

BEFORE change of coordinates:
Rotation-translation component of the resistance tensor:
[[  0.          0.          5.9759693]
 [  0.          0.        -10.158139 ]
 [ -5.5253577   9.873121    0.       ]] 

OLD default param values: ['radius0', 'radius1', 'radius2', 'xref0', 'xref1', 'xref2'] [1.   0.75 0.5  0.   0.   0.  ]
NEW default param values: ['radius0', 'radius1', 'radius2', 'xref0', 'xref1', 'xref2'] [ 1.          0.75        0.5        -0.4227919  -0.24754643  0.        ]

AFTER change of coordinates:
Rotation-translation component of the resistance tensor:
[[ 0.          0.         -0.12446499]
 [ 0.          0.          0.26097277]
 [-0.12446499  0.26097703  0.        ]]
R_mean is now symmetric


We repeat the verification for the **mobility** tensor, shifting the assembly so the centre of mobility coincides with the reference point (`xref` set to $-\mathbf{x}_{CM}$).

In [9]:
# Assert the mobility tensor with x_cm the reference point
print("BEFORE change of coordinates:")
M_translation_rotation = M_rigid[3:,:3]
print("Rotation-translation component of the mobility tensor:")
print(M_translation_rotation, "\n")

# Shift the assembly so the centre of mobility coincides with the reference point
mybody.set_design_defaults(new_dict={"xref0": -x_cm[0], "xref1": -x_cm[1], "xref2": -x_cm[2]})

M_rigid = np.array(mybody.compute_rigid_mobility())

print("\nAFTER change of coordinates:")
M_translation_rotation = M_rigid[3:,:3]
print("Rotation-translation component of the mobility tensor:")
print(M_translation_rotation)

# Check if the matrix is symmetric
if np.allclose(M_translation_rotation, M_translation_rotation.T, atol=1E-5):
    print("M_mean is now symmetric")
else:
    raise ValueError("M_mean is not symmetric after change of coordinates")

BEFORE change of coordinates:
Rotation-translation component of the mobility tensor:
[[ 0.0000000e+00  0.0000000e+00  1.2667943e-04]
 [ 0.0000000e+00  0.0000000e+00 -2.0915063e-04]
 [ 8.9041554e-05 -1.7846700e-04  0.0000000e+00]] 

OLD default param values: ['radius0', 'radius1', 'radius2', 'xref0', 'xref1', 'xref2'] [ 1.          0.75        0.5        -0.4227919  -0.24754643  0.        ]
NEW default param values: ['radius0', 'radius1', 'radius2', 'xref0', 'xref1', 'xref2'] [ 1.          0.75        0.5        -0.42194483 -0.24658029 -0.        ]

AFTER change of coordinates:
Rotation-translation component of the mobility tensor:
[[ 0.          0.          0.00010407]
 [ 0.          0.         -0.00019165]
 [ 0.00010408 -0.00019165  0.        ]]
M_mean is now symmetric
